# Sequential foraminiferal chamber growth

A 2D phase-field model of a foraminifer growing chamber by chamber from a fixed seed. Each new chamber grows from an aperture on its immediate predecessor to a prescribed area, stays outside existing chambers, and is attracted toward older walls. Older chambers never move. Time is measured in formation times: each chamber grows over one time unit. Sizes, timing and shape parameters are illustrative, not fitted biological values. See `README.md` for the equations, parameters and limitations.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
from foram_colony import ColonyParameters, simulate_colony, save_colony, render_colony
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

## Bilocular seed

The seed has two distinct connected compartments, labelled 1 and 2. Growth starts at chamber 3, attached to chamber 2. The first compartment keeps its generating disc; the second occupies its own disc outside the first, so their interiors don't overlap. The second compartment's area therefore depends on both the radius ratio and the separation.

The pair lies along the x axis. Set `second_piece_radius_ratio=0.85` for a smaller second disc. `new_chambers` and `chamber_areas` exclude both seed compartments.

In [ ]:
bilocular_parameters = ColonyParameters(
    seed_mode="bilocular",
    central_radius=0.21,
    second_piece_radius_ratio=1.10,
    seed_separation_fraction=0.60,
    new_chambers=6,
    adhesion_strength=2.5,
    turning_angle_degrees=65.0,
)
bilocular_result = simulate_colony(bilocular_parameters, device=device)
bilocular_metrics = save_colony(bilocular_result, Path("output/bilocular"))
render_colony(bilocular_result, Path("output/bilocular"))
bilocular_metrics["seed_reports"]

In [ ]:
display(Image(filename="output/bilocular/chamber_growth.gif"))

## Unilocular seed

The seed is a single central chamber 1; growth starts at chamber 2.

In [ ]:
unilocular_parameters = ColonyParameters(
    seed_mode="unilocular",
    new_chambers=6,
    central_radius=0.30,
    first_chamber_area=0.28,
    chamber_area_ratio=1.16,
    turning_angle_degrees=65.0,
    adhesion_strength=2.5,
    adhesion_range=0.14,
    spacing=0.025,
)
unilocular_result = simulate_colony(unilocular_parameters, device=device)
unilocular_metrics = save_colony(unilocular_result, Path("output/unilocular"))
render_colony(unilocular_result, Path("output/unilocular"))
unilocular_metrics["reports"]

In [ ]:
display(Image(filename="output/unilocular/chamber_growth.gif"))
display(Image(filename="output/unilocular/growth_summary.png"))

## Controls and specified chamber sizes

`adhesion_strength=0` switches off attraction while keeping the parent attachment. `turning_angle_degrees` sets the guide for next-aperture placement. Pass one final area per new chamber with `chamber_areas`. The frozen (pixel) chamber areas differ from the diffuse areas by at most half a pixel area.

In [ ]:
custom_result = simulate_colony(ColonyParameters(new_chambers=3, chamber_areas=(0.30, 0.39, 0.48)),
                                device=device)
save_colony(custom_result, Path("output/custom"))
render_colony(custom_result, Path("output/custom"))
display(Image(filename="output/custom/growth_summary.png"))